<a href="https://colab.research.google.com/github/ivansst773/Aprendizaje_de_Maquina/blob/main/TallerRL_TAM_2025_1/Te_damos_la_bienvenida_a_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalación de dependencias
!apt-get update -y
!apt-get install -y xvfb ffmpeg freeglut3-dev
!pip install tensorflow==2.15.0 tensorflow-probability==0.22.0 tf-agents==0.17.0 gymnasium pyvirtualdisplay imageio --quiet

# Imports
import tensorflow as tf
from tf_agents.environments import suite_gymnasium, tf_py_environment
from tf_agents.agents.ddpg import ddpg_agent
from tf_agents.networks import actor_network, critic_network
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.drivers import dynamic_step_driver
from tf_agents.policies import random_tf_policy
from tf_agents.utils import common
import matplotlib.pyplot as plt
import imageio
import numpy as np

# Asegúrate de que TF Agents use Keras en el mismo contexto que TensorFlow
# Esto es importante para evitar el AttributeError
tf.compat.v1.enable_v2_behavior()


# Crear entorno
env_name = "Pendulum-v1"
train_env = tf_py_environment.TFPyEnvironment(suite_gymnasium.load(env_name))
eval_env = tf_py_environment.TFPyEnvironment(suite_gymnasium.load(env_name))

# Redes neuronales
fc_layer_params = (400, 300)
actor_net = actor_network.ActorNetwork(train_env.observation_spec(), train_env.action_spec(), fc_layer_params=fc_layer_params)
critic_net = critic_network.CriticNetwork((train_env.observation_spec(), train_env.action_spec()), fc_layer_params=fc_layer_params)

# Optimizadores
actor_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
critic_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

# Agente DDPG
global_step = tf.compat.v1.train.get_or_create_global_step()
agent = ddpg_agent.DdpgAgent(
    train_env.time_step_spec(),
    train_env.action_spec(),
    actor_network=actor_net,
    critic_network=critic_net,
    actor_optimizer=actor_optimizer,
    critic_optimizer=critic_optimizer,
    ou_stddev=0.2,
    ou_damping=0.15,
    target_update_tau=0.05,
    target_update_period=5,
    td_errors_loss_fn=common.element_wise_squared_loss,
    train_step_counter=global_step
)
agent.initialize()

# Política aleatoria
random_policy = random_tf_policy.RandomTFPolicy(train_env.time_step_spec(), train_env.action_spec())

# Buffer de repetición
replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    data_spec=agent.collect_data_spec,
    batch_size=train_env.batch_size,
    max_length=100000
)

# Driver de recolección
collect_driver = dynamic_step_driver.DynamicStepDriver(
    train_env,
    agent.collect_policy,
    observers=[replay_buffer.add_batch],
    num_steps=1
)

# Función para evaluar el agente
def compute_avg_return(environment, policy, num_episodes=10):
    total_return = 0.0
    for _ in range(num_episodes):
        time_step = environment.reset()
        policy_state = policy.get_initial_state(environment.batch_size)
        episode_return = 0.0
        while not time_step.is_last():
            action_step = policy.action(time_step, policy_state)
            time_step = environment.step(action_step.action)
            episode_return += time_step.reward.numpy()[0]
        total_return += episode_return
    return total_return / num_episodes

# Entrenamiento
returns = []
num_iterations = 50000
for _ in range(num_iterations):
    collect_driver.run()
    # Asegúrate de que replay_buffer.gather_all() no devuelva un batch vacío si no hay suficientes datos
    if replay_buffer.num_elements() >= train_env.batch_size: # O un valor mínimo adecuado
        experience, unused_info = replay_buffer.sample(train_env.batch_size)
        agent.train(experience)
        # Limpia el buffer después de cada paso de entrenamiento, o cada cierto número de pasos
        replay_buffer.clear() # Puedes ajustar cuándo limpiar esto para mejorar la estabilidad

    if global_step.numpy() % 1000 == 0:
        avg_return = compute_avg_return(eval_env, agent.policy, 10)
        returns.append(avg_return)
        print(f"Step: {global_step.numpy()}, Avg Return: {avg_return}")

# Gráfica del reward
plt.plot(returns)
plt.title("Reward esperado")
plt.xlabel("Iteraciones (x1000)")
plt.ylabel("Return promedio")
plt.grid()
plt.show()

# Generar video del agente
def generate_video(policy, filename="agent_video.mp4"):
    frames = []
    time_step = eval_env.reset()
    policy_state = policy.get_initial_state(eval_env.batch_size)
    for _ in range(200):
        action_step = policy.action(time_step, policy_state)
        time_step = eval_env.step(action_step.action)
        # Asegúrate de que el renderizado es correcto para el entorno
        # 'rgb_array' es el modo de renderizado estándar para obtener una imagen
        frame = eval_env.pyenv.envs[0].render()
        frames.append(frame)
    imageio.mimsave(filename, frames, fps=30)

generate_video(agent.policy, "trained_agent.mp4")
generate_video(random_policy, "random_agent.mp4")
